# Pipeline Học Máy Dữ Liệu Bảng

Notebook điều phối chính cho pipeline học máy dữ liệu bảng, bao gồm:
- Data ingestion từ URL công khai
- Exploratory Data Analysis (EDA)
- Preprocessing (imputation, encoding, scaling)
- Feature engineering (PCA, artifact export)
- Model training (traditional ML + MLP)
- Evaluation và visualization

**Team Members:** [Thêm tên thành viên]

**Date:** March 2026

## 1. Setup and Imports

In [ ]:
# Import thư viện cơ bản
import sys
import os
import numpy as np
import pandas as pd

# Thêm modules vào path
sys.path.append('../modules')

# Import các modules của project
# TODO: Uncomment sau khi implement các modules
from data_loader import load_data_pipeline
# from eda import run_full_eda
from preprocessing import preprocess_pipeline
# from features import apply_pca, save_features
# from models import train_logistic_regression, train_svm, train_random_forest, train_mlp
# from evaluation import evaluate_model, compare_models

print("✓ Imports completed")

✓ Imports completed


## 2. Configuration Dictionary

Centralized configuration cho toàn bộ pipeline. Thay đổi config ở đây để thử nghiệm các cấu hình khác nhau.

In [2]:
import sys
import os

# 1. Thêm thư mục gốc của dự án vào đường dẫn hệ thống để import được config.py
sys.path.append(os.path.abspath('..'))

# 2. Đổi thư mục làm việc hiện tại về thư mục gốc của dự án.
# Điều này rất quan trọng vì các đường dẫn trong config.py đang được viết ở dạng 
# tương đối từ thư mục gốc (vd: 'data/raw_data.csv', 'reports/eda/')
os.chdir(os.path.abspath('..'))

# 3. Import biến CONFIG đã được định nghĩa sẵn trong config.py
from config import CONFIG

print("✓ Configuration loaded từ file config.py ở thư mục gốc")

✓ Configuration loaded từ file config.py ở thư mục gốc


## 3. Pipeline Orchestration

### Pipeline Flow
```
Data Ingestion → EDA → Preprocessing → Feature Engineering → Model Training → Evaluation
```

### 3.1 Data Ingestion

Tải dữ liệu từ URL công khai và load vào DataFrame.

In [3]:
# Import hàm đọc file trực tiếp từ module
from data_loader import create_dataframe

# Lấy đường dẫn file từ cấu hình CONFIG (hoặc bạn tự truyền đường dẫn 'data/hotel_bookings.csv')
file_path = CONFIG['data']['file_path'] 

# Đọc thẳng file local thành DataFrame và lấy metadata
df, metadata = create_dataframe(file_path)

# In ra các thông số cơ bản của dữ liệu
print(f"✓ Data loaded directly from local: {df.shape}")
print(f"  - Rows: {metadata['n_rows']}")
print(f"  - Columns: {metadata['n_columns']}")
print(f"  - Missing values: {metadata['total_missing']}")

[Data Loader] Đọc dữ liệu thành công - shape: (119390, 32)
✓ Data loaded directly from local: (119390, 32)
  - Rows: 119390
  - Columns: 32
  - Missing values: 129425


### 3.2 Exploratory Data Analysis

Phân tích mô tả và trực quan hóa dữ liệu.

In [4]:
import os
# Import các hàm đã được định nghĩa trong module eda
from modules.eda import generate_eda_report, plot_missing_values

# Lấy đường dẫn thư mục lưu từ CONFIG (hoặc dùng mặc định 'reports/eda' nếu chưa có)
output_dir = CONFIG['eda']['output_dir'] if 'eda' in CONFIG and 'output_dir' in CONFIG['eda'] else 'reports/eda'
os.makedirs(output_dir, exist_ok=True)

print("=== BẮT ĐẦU EDA ===")
# 1. Gọi hàm generate_eda_report để phân tích toàn bộ dữ liệu
report = generate_eda_report(df, output_dir=output_dir)

# 2. Lấy kết quả thống kê các cột agent, company, children đã được tính toán trong module
target_missing = report["missing_values_analysis"]["target_columns_missing"]

print("\n[THỐNG KÊ SỐ LƯỢNG GIÁ TRỊ TRỐNG]")
for col, missing_count in target_missing.items():
    print(f"- Cột '{col}': {missing_count} giá trị thiếu")

# 3. Trực quan hoá dữ liệu thiếu (lưu thành biểu đồ)
plot_path = os.path.join(output_dir, "missing_values_chart.png")
plot_missing_values(df, output_path=plot_path)

print("\n✓ EDA completed")

=== BẮT ĐẦU EDA ===
[EDA] Tạo báo cáo thống kê mô tả
[EDA] Đã lưu báo cáo thống kê mô tả tại: reports/eda/eda_report.json

[THỐNG KÊ SỐ LƯỢNG GIÁ TRỊ TRỐNG]
- Cột 'children': 4 giá trị thiếu
- Cột 'country': 488 giá trị thiếu
- Cột 'agent': 16340 giá trị thiếu
- Cột 'company': 112593 giá trị thiếu
[EDA] Vẽ biểu đồ missing values và lưu vào reports/eda/missing_values_chart.png
[EDA] Đang vẽ biểu đồ missing values và lưu vào reports/eda/missing_values_chart.png...
[EDA] Hoàn tất lưu biểu đồ tại: reports/eda/missing_values_chart.png

✓ EDA completed


/Users/phutuenguyen/HCMUT/252/Machine Learning/btl/code/ML_Assignment_K22_HCMUT_GROUP_6/modules/eda.py:169: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=missing_counts.index,


### 3.3 Data Preprocessing

Tiền xử lý: Imputation → Encoding → Scaling

In [5]:
# Gọi hàm preprocess_pipeline để thực hiện imputation, encoding và scaling
X, y, preprocessing_metadata = preprocess_pipeline(
    df,
    CONFIG['data']['target_column'],
    CONFIG['preprocessing']
)

print(f"✓ Preprocessing completed")
print(f"  - Features shape: {X.shape}")
print(f"  - Target shape: {y.shape}")

NameError: name 'preprocess_pipeline' is not defined

### 3.4 Feature Engineering

Apply PCA và lưu features artifacts.

In [ ]:
# TODO: Implement feature engineering
# if CONFIG['features']['pca']['enabled']:
#     X_pca, pca_info = apply_pca(X, CONFIG['features']['pca'])
#     print(f"✓ PCA completed")
#     print(f"  - Original dimensions: {X.shape[1]}")
#     print(f"  - Reduced dimensions: {X_pca.shape[1]}")
# else:
#     X_pca = X
# 
# # Lưu features
# save_features(X_pca, y, CONFIG['features']['output'])
# print(f"✓ Features saved")

print("TODO: Implement feature engineering step")

### 3.5 Model Training

Huấn luyện các models: Logistic Regression, SVM, Random Forest, MLP.

In [ ]:
# TODO: Implement model training
# from sklearn.model_selection import train_test_split
# 
# # Split data
# X_train, X_test, y_train, y_test = train_test_split(
#     X_pca, y,
#     test_size=CONFIG['models']['train_test_split']['test_size'],
#     random_state=CONFIG['models']['train_test_split']['random_state']
# )
# 
# results = {}
# 
# # Train models theo config
# # TODO: Add training code for each enabled model
# 
# print("✓ All models trained")

print("TODO: Implement model training step")

### 3.6 Model Evaluation

Đánh giá và so sánh các models.

In [ ]:
# TODO: Implement model evaluation
# for model_name, model_data in results.items():
#     eval_results = evaluate_model(
#         y_test,
#         model_data['predictions'],
#         model_data.get('probabilities'),
#         model_name,
#         CONFIG['evaluation']['output_dir'],
#         CONFIG['evaluation']
#     )
#     print(f"\n{model_name}:")
#     for metric, value in eval_results['metrics'].items():
#         print(f"  {metric}: {value:.4f}")
# 
# # So sánh models
# compare_models(results, f"{CONFIG['evaluation']['output_dir']}/model_comparison.png")
# print("\n✓ Evaluation completed")

print("TODO: Implement model evaluation step")

## 4. Summary và Next Steps

### Pipeline Summary
- ✓ Data ingestion từ URL công khai
- ✓ EDA với visualizations
- ✓ Preprocessing với configurable options
- ✓ Feature engineering với PCA
- ✓ Model training (traditional + deep learning)
- ✓ Evaluation và comparison

### Next Steps
1. Implement các TODO trong modules/
2. Test pipeline với dataset thực tế
3. Tune hyperparameters
4. Tổng hợp kết quả vào báo cáo

### Team Notes
- [Thêm ghi chú cho team]
- [Phân công công việc]
- [Deadline và milestones]